# 05c — Análise de Ablação + KNN Melhorado

Este notebook realiza **dois experimentos críticos**:

1. **Ablação**: Remove cada feature uma de cada vez para identificar possível data leakage
2. **KNN melhorado**: Tuning agressivo com `weights='distance'` e busca de threshold ótimo

## Motivação

- A árvore de decisão teve R² ≈ 0,998 no teste (suspeitosamente alto) → pode indicar leakage
- KNN teve F1 baixo (verdadeiro, por sensibilidade a desbalanceamento)
- Objetivo: **quantificar** o impacto de cada feature e **melhorar** KNN com tuning agressivo

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import INTERIM_DIR
from src.ablacao import run_ablation_experiment
from src.knn_melhorado import run_knn_improved, visualize_knn_results
from sklearn.preprocessing import StandardScaler

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

RANDOM_STATE = 42

## 1. Preparação de dados

In [ ]:
# Carrega dataset
path = INTERIM_DIR / "dataset_analitico_com_scores.parquet"
df = pd.read_parquet(path)

print(f"Dataset: {df.shape}")
print(f"\nFold distribution:")
print(df['fold'].value_counts())

df_train = df[df['fold'] == 'train'].copy()
df_test = df[df['fold'] == 'test'].copy()

y_train = df_train['anomalia_label'].astype(int)
y_test = df_test['anomalia_label'].astype(int)

print(f"\nAnomalias no treino: {y_train.sum()} / {len(y_train)} ({y_train.mean():.2%})")
print(f"Anomalias no teste:  {y_test.sum()} / {len(y_test)} ({y_test.mean():.2%})")

## 2. Análise de Ablação

**Hipótese**: Se remover a feature correta causa queda significativa, está confirmado o leakage.

In [ ]:
# Define features base
base_features = [
    'log_valor_licitacao',
    'n_participantes',
    'n_itens',
    'hhi',
    'top1_share',
    'valor_total_itens',
]

print("Features a analisar:")
for i, f in enumerate(base_features, 1):
    print(f"  {i}. {f}")

In [ ]:
# Executa ablação
ablation_results = []

# Baseline com todas
print("\n" + "="*80)
print("BASELINE: Todas as features")
print("="*80)

result_baseline = run_ablation_experiment(
    feature_set_name="Todas (baseline)",
    features=base_features,
    X_train=df_train,
    y_train=y_train,
    X_test=df_test,
    y_test=y_test,
)
ablation_results.append(result_baseline)

print(f"\n  KNN F1: {result_baseline['knn_f1']:.4f}")
print(f"  DT F1:  {result_baseline['dt_f1']:.4f}")
print(f"  RF F1:  {result_baseline['rf_f1']:.4f}")

In [ ]:
# Ablação: remover cada feature
for feature_to_remove in base_features:
    remaining = [f for f in base_features if f != feature_to_remove]
    
    print(f"\n{'='*80}")
    print(f"ABLAÇÃO: Removendo '{feature_to_remove}'")
    print(f"Restante: {remaining}")
    print(f"{'='*80}")
    
    result = run_ablation_experiment(
        feature_set_name=f"Sem {feature_to_remove}",
        features=remaining,
        X_train=df_train,
        y_train=y_train,
        X_test=df_test,
        y_test=y_test,
    )
    ablation_results.append(result)
    
    print(f"\n  KNN F1: {result['knn_f1']:.4f} (Δ {result['knn_f1'] - result_baseline['knn_f1']:+.4f})")
    print(f"  DT F1:  {result['dt_f1']:.4f} (Δ {result['dt_f1'] - result_baseline['dt_f1']:+.4f})")
    print(f"  RF F1:  {result['rf_f1']:.4f} (Δ {result['rf_f1'] - result_baseline['rf_f1']:+.4f})")

## 3. Tabela Comparativa de Ablação

In [ ]:
# Prepara DataFrame de ablação
ablation_df = pd.DataFrame(ablation_results)

# Cálcula mudanças em relação ao baseline
baseline_knn_f1 = ablation_df.iloc[0]['knn_f1']
baseline_dt_f1 = ablation_df.iloc[0]['dt_f1']
baseline_rf_f1 = ablation_df.iloc[0]['rf_f1']

ablation_df['knn_f1_delta'] = ablation_df['knn_f1'] - baseline_knn_f1
ablation_df['dt_f1_delta'] = ablation_df['dt_f1'] - baseline_dt_f1
ablation_df['rf_f1_delta'] = ablation_df['rf_f1'] - baseline_rf_f1

# Display
display_cols = [
    'feature_set', 'num_features',
    'knn_f1', 'knn_f1_delta',
    'dt_f1', 'dt_f1_delta',
    'rf_f1', 'rf_f1_delta',
]

print("\nRESUMO DA ABLAÇÃO")
print("="*120)
print(ablation_df[display_cols].to_string(index=False))

# Identifica feature mais impactante por modelo
print("\n" + "="*120)
print("FEATURE MAIS IMPACTANTE (queda de F1):")
print("="*120)

ablation_df_exclude_baseline = ablation_df.iloc[1:]  # exclude baseline

most_impact_knn_idx = ablation_df_exclude_baseline['knn_f1_delta'].idxmin()
most_impact_dt_idx = ablation_df_exclude_baseline['dt_f1_delta'].idxmin()
most_impact_rf_idx = ablation_df_exclude_baseline['rf_f1_delta'].idxmin()

print(f"\nKNN:")
print(f"  Feature: {ablation_df.iloc[most_impact_knn_idx]['feature_set']}")
print(f"  Queda: {ablation_df.iloc[most_impact_knn_idx]['knn_f1_delta']:.4f}")

print(f"\nDecision Tree:")
print(f"  Feature: {ablation_df.iloc[most_impact_dt_idx]['feature_set']}")
print(f"  Queda: {ablation_df.iloc[most_impact_dt_idx]['dt_f1_delta']:.4f}")

print(f"\nRandom Forest:")
print(f"  Feature: {ablation_df.iloc[most_impact_rf_idx]['feature_set']}")
print(f"  Queda: {ablation_df.iloc[most_impact_rf_idx]['rf_f1_delta']:.4f}")

In [ ]:
# Salva ablação
out_path = INTERIM_DIR / "ablacao_resultados.parquet"
ablation_df.to_parquet(out_path, index=False)
print(f"\n✓ Resultados de ablação salvos em: {out_path}")

## 4. Visualização de Impacto de Features

In [ ]:
# Plota impacto de cada feature
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ablation_exclude = ablation_df.iloc[1:]  # exclude baseline

# KNN
axes[0].barh(range(len(ablation_exclude)), ablation_exclude['knn_f1_delta'].values, color='steelblue')
axes[0].set_yticks(range(len(ablation_exclude)))
axes[0].set_yticklabels([s.replace('Sem ', '') for s in ablation_exclude['feature_set'].values], fontsize=10)
axes[0].set_xlabel('Δ F1 Score', fontsize=11)
axes[0].set_title('KNN - Impacto da Remoção de Features', fontsize=12, fontweight='bold')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
axes[0].grid(True, alpha=0.3, axis='x')

# Decision Tree
axes[1].barh(range(len(ablation_exclude)), ablation_exclude['dt_f1_delta'].values, color='coral')
axes[1].set_yticks(range(len(ablation_exclude)))
axes[1].set_yticklabels([s.replace('Sem ', '') for s in ablation_exclude['feature_set'].values], fontsize=10)
axes[1].set_xlabel('Δ F1 Score', fontsize=11)
axes[1].set_title('Decision Tree - Impacto da Remoção de Features', fontsize=12, fontweight='bold')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].grid(True, alpha=0.3, axis='x')

# Random Forest
axes[2].barh(range(len(ablation_exclude)), ablation_exclude['rf_f1_delta'].values, color='mediumseagreen')
axes[2].set_yticks(range(len(ablation_exclude)))
axes[2].set_yticklabels([s.replace('Sem ', '') for s in ablation_exclude['feature_set'].values], fontsize=10)
axes[2].set_xlabel('Δ F1 Score', fontsize=11)
axes[2].set_title('Random Forest - Impacto da Remoção de Features', fontsize=12, fontweight='bold')
axes[2].axvline(0, color='black', linestyle='--', linewidth=1)
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(INTERIM_DIR / 'ablacao_impacto_features.png', dpi=300, bbox_inches='tight')
print(f"✓ Gráfico salvo em: {INTERIM_DIR / 'ablacao_impacto_features.png'}")
plt.show()

## 5. KNN Melhorado - Tuning Agressivo

**Estratégia**:
- Testar k ∈ {1, 3, 5, 7, 15, 25}
- Testar weights ∈ {'uniform', 'distance'}
- Otimizar por F1 (classe rara)
- Buscar threshold ótimo com predict_proba

In [ ]:
# Prepara dados padronizados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train[base_features].fillna(0))
X_test_scaled = scaler.transform(df_test[base_features].fillna(0))

print("Features padronizadas com StandardScaler")

In [ ]:
# Executa KNN melhorado
knn_result = run_knn_improved(X_train_scaled, y_train, X_test_scaled, y_test, random_state=RANDOM_STATE)

## 6. Visualizações do KNN

In [ ]:
# Visualiza resultados
fig = visualize_knn_results(knn_result, y_test, output_dir=INTERIM_DIR)

In [ ]:
# Tabela de top k values
cv_df = knn_result['cv_history']

print("\nTop 10 configurações de KNN (por F1):")
print("="*80)
top_configs = cv_df.nlargest(10, 'f1_mean')[['k', 'weights', 'f1_mean', 'f1_std', 'roc_auc_mean']]
print(top_configs.to_string(index=False))

## 7. Comparação: Antes vs Depois do Tuning

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, auc, precision_recall_curve

# KNN padrão (k=5, weights='uniform')
from sklearn.neighbors import KNeighborsClassifier
knn_default = KNeighborsClassifier(n_neighbors=5, weights='uniform')
knn_default.fit(X_train_scaled, y_train)
y_pred_default = knn_default.predict(X_test_scaled)
y_proba_default = knn_default.predict_proba(X_test_scaled)[:, 1]

# KNN melhorado (com threshold ótimo)
y_pred_improved = knn_result['y_pred_optimal']
y_proba_improved = knn_result['y_proba']

comparison_data = {
    'Métrica': ['F1', 'Precisão', 'Recall', 'ROC-AUC', 'PR-AUC'],
    'KNN Default': [
        f1_score(y_test, y_pred_default),
        precision_score(y_test, y_pred_default),
        recall_score(y_test, y_pred_default),
        roc_auc_score(y_test, y_proba_default),
        auc(*precision_recall_curve(y_test, y_proba_default)[:2])
    ],
    'KNN Melhorado': [
        f1_score(y_test, y_pred_improved),
        precision_score(y_test, y_pred_improved),
        recall_score(y_test, y_pred_improved),
        roc_auc_score(y_test, y_proba_improved),
        auc(*precision_recall_curve(y_test, y_proba_improved)[:2])
    ]
}

comp_df = pd.DataFrame(comparison_data)
comp_df['Δ (Melhoria)'] = comp_df['KNN Melhorado'] - comp_df['KNN Default']

print("\nCOMPARAÇÃO: KNN Default vs KNN Melhorado")
print("="*80)
print(comp_df.to_string(index=False))

# Salva comparação
comp_df.to_parquet(INTERIM_DIR / 'knn_comparacao_before_after.parquet', index=False)

## Conclusões

**De Ablação:**
- Identificar qual feature tem maior impacto (confirmar possível leakage)
- Se Decision Tree cai muito com uma remoção específica → leakage confirmado

**De KNN Melhorado:**
- Melhor configuração de k e weights descoberta
- Threshold ótimo encontrado (se diferente de 0.5, aumenta recall de anomalias)
- Comparação quantificada: antes vs depois do tuning